# 1. Matching: right heart catheterization (RHC) data

Does right heart catheterization (`treatment`) raise the risk of death (`died`)? Patients who got RHC were
sicker, so compare like with like: match treated and control patients on covariates, check balance with
standardized mean differences (SMD), then compare outcomes within the matched pairs.

1. greedy matching on Mahalanobis distance (`Matching::Match`)
2. propensity score matching with `MatchIt`
3. propensity score matching by hand: logistic regression, then greedy matching on logit(PS) with a caliper

In [ ]:
source("setup.R")
use_packages("tableone", "Matching", "MatchIt")

mydata <- load_rhc()
xvars <- rhc_xvars
head(mydata)

## Table 1: balance before matching

In [ ]:
table1 <- CreateTableOne(vars = xvars, strata = "treatment", data = mydata, test = FALSE)
print(table1, smd = TRUE)

## Greedy matching on Mahalanobis distance

In [ ]:
greedymatch <- Match(Tr = mydata$treatment, M = 1, X = mydata[xvars], replace = FALSE)
matched <- matched_rows(greedymatch, mydata)

# balance after matching
matchedtab1 <- CreateTableOne(vars = xvars, strata = "treatment", data = matched, test = FALSE)
print(matchedtab1, smd = TRUE)

In [ ]:
# outcome analysis: within-pair differences
y_trt <- matched$died[matched$treatment == 1]
y_con <- matched$died[matched$treatment == 0]
diffy <- y_trt - y_con

# paired t-test
t.test(diffy)

# McNemar test on the 2x2 table of pair outcomes
pairs <- table(y_trt, y_con)
pairs
mcnemar.test(pairs)

## Matching on propensity scores with MatchIt

`matchit` fits the propensity score model (logistic regression) itself.

In [ ]:
m.out <- matchit(treatment ~ ARF + CHF + Cirr + colcan + Coma + MOSF + sepsis + age + female + meanbp1,
                 method = "nearest", data = mydata)
summary(m.out)
plot(m.out, type = "jitter", interactive = FALSE)
plot(m.out, type = "hist")

## Propensity score matching by hand

Fit the propensity score model, then greedy-match on logit(PS) with a caliper of 0.2.
(The lecture's model also uses `aps1`, which is not in this version of the data.)

In [ ]:
psmodel <- glm(treatment ~ ARF + CHF + Cirr + colcan + Coma + lungcan + MOSF + sepsis + age + female + meanbp1,
               family = binomial(), data = mydata)
summary(psmodel)
pscore <- psmodel$fitted.values

In [ ]:
psmatch <- Match(Tr = mydata$treatment, M = 1, X = logit(pscore), replace = FALSE, caliper = .2)
matched <- matched_rows(psmatch, mydata)

matchedtab1 <- CreateTableOne(vars = xvars, strata = "treatment", data = matched, test = FALSE)
print(matchedtab1, smd = TRUE)

# outcome analysis
y_trt <- matched$died[matched$treatment == 1]
y_con <- matched$died[matched$treatment == 0]
t.test(y_trt - y_con)